# 23 — Malignant annotation on the v2 atlas: ALICE TCR clonality + inferCNV

Two independent lines of evidence for the same question, on the `21_reannotation` **v2** re-annotation
`data/atlas_joint/skin_T_annotated.h5ad` (539,916 T cells / 146 donors / 174 samples / 14 studies,
against v1's 395,233 / ~60 / 7 studies):

| | steps | unit | output |
|---|---|---|---|
| **TCR clonality** | 1–4 | ALICE per donor, reported per sample | `tables/skin_tcr_clonality_per_sample.csv` |
| **CNV** | 5–9 | inferCNV per donor | `skin_T_malignancy_v5.parquet` |

The TCR half answers one question — **which samples carry a clonal CD4 population, how large, and
what kind of sample is it** — and ends in a single table. (It used to run three ALICE analyses:
the cross-patient convergence test returned 0 hits out of 30 pooled founders and the reactive-CD8
pass answers a different question about the TIL infiltrate; both were cut and are recoverable from
git history.)

The object already carries TCR obs (`tra_cdr3`/`trb_cdr3`/`has_tcr`), MrVI latents (`X_mrvi_u`), and
raw counts, so it *replaces* the old hand-assembled `skin_T_tcr_malig_v2.h5ad` — no separate curated
object is written. Helpers are reused verbatim (`alice_helpers`, `skin_T_cnv_helpers`).

Seven of the 14 v2 skin cohorts carry no V(D)J at all, so Step 1 gates the TCR and CNV arms
**separately**: `obs["tcr_cohort"]` marks which donors have a repertoire, and no downstream step is
allowed to read missing TCR data as a negative TCR call. Caches are versioned by filename only —
TCR/ALICE are `_v4`, CNV is `_v5`; the v3 and v4 CNV caches stay on disk (`24_subclone_tcr_signaling` reads v4, `old/37_semantic_clonality_transfer` reads v3).

> **HEAVY** cells (TCR-object build, OLGA Pgen sweep, per-donor inferCNV, per-arm inferCNV) run on the
> GPU/compute kernel — not the login node.

In [ ]:
# ============================================================
# Parameters — inputs, cohort gates, ALICE + inferCNV knobs.
# ============================================================
import hashlib, json
from pathlib import Path


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data from {start}")


NB_DIR = _resolve_nb_dir(); print("NB_DIR =", NB_DIR)
OUT_DIR = NB_DIR / "data" / "atlas_joint"
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
TAB_DIR = NB_DIR / "tables"; TAB_DIR.mkdir(exist_ok=True)

# ---- inputs (the nb10b re-annotation on the **v2** atlas) ----
# v2 is a different cohort, not a refresh: 539,916 T cells / 146 donors / 174 samples / 14 studies,
# against v1's 395,233 / ~60 / 7 studies. Every cache below is versioned by filename only, so each
# one that survived from the v1 run would be loaded silently and the notebook would report v1
# numbers on a v2 input. Hence the bump: TCR object + ALICE -> v4, CNV -> v5. Delete nothing —
# nb31 / nb37 still read skin_T_malignancy_v3.
OBJ     = OUT_DIR / "skin_T_annotated.h5ad"              # nb10b v2 re-annotation (cell_type_T)
TCR_OBJ = OUT_DIR / "skin_T_tcr_annotated_v4.h5ad"       # fresh TCR-complete cache (built on first run)
LI_TCR  = NB_DIR / "data" / "Li2024_atlas" / "li2024_tcr_malignancy.parquet"
GTF           = NB_DIR / "data" / "cache" / "Homo_sapiens.GRCh38.110.chr.gtf.gz"
INTEGRATED_H5 = NB_DIR / "data" / "Integrated_CTCL_skincellatlas_final_portal_tags.h5ad"

# ---- clonality rule + cohort gates (see Step 1 / Step 4) ----
FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN = 0.05, 1.33, 2
TCR_MIN_CELLS = 300   # ALICE arm: a repertoire this thin cannot calibrate an OLGA null
CNV_MIN_T     = 200   # CNV arm: fewer T than this cannot anchor a CNV cluster (MIN_CLUSTER_FRAC)
# v2 vocabulary: cell_type_T = {CD4, CD8, CD4_Treg} (v1's UNK is gone — nb10b §11.5 resolved it).
# CD4_Treg stays OUT of the clonal cohort, as v1's "Tregs" did: reactive Tregs expand clonally too,
# so a Treg-labelled dominant clone is not evidence of tumour. They keep a CNV score, so a FOXP3+
# tumour clone can still surface through the CNV arm.
CD4_T_TYPES = ["CD4"]
DROP_ENTITIES = {"MF_gamma_delta", "CD8_aggressive_epidermotropic_CTCL"}

# ---- ALICE params ----
ALPHA = 0.05
Q = None              # None -> calibrate per repertoire from the null bulk
SEED = 0

# ---- TCR outputs (v4 = built on the nb10b v2 re-annotation) ----
ALICE_CD4_PARQUET = OUT_DIR / "alice_cd4_per_donor_v4.parquet"   # per-clonotype ALICE result
ALICE_MAL_PARQUET = OUT_DIR / "alice_malignancy_v4.parquet"      # per-cell tcr_clonal
CLONALITY_CSV     = TAB_DIR / "skin_tcr_clonality_per_sample.csv"  # <- the conclusive table

# ---- CNV outputs (v5 = the v4 null-anchored arm-consensus caller, unchanged, rebuilt on v2).
#      v3 (nb31/nb37) and v4 (the v1-cohort run) both stay on disk; neither is overwritten. ----
OUT_PARQUET = OUT_DIR / "skin_T_malignancy_v5.parquet"           # per-cell malignancy (v5)
SUMMARY_CSV = TAB_DIR / "skin_cnv_v5_summary.csv"                # <- the conclusive CNV table
# CNV_CACHE / ARM_CACHE (v5) are defined alongside the estimator in Step 5.

In [ ]:
import sys, gc, importlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sys.path.insert(0, str(NB_DIR / "helpers"))
import atlas_join_helpers as H
import skin_T_cnv_helpers as C
import alice_helpers as A
for _m in (H, C, A):
    importlib.reload(_m)
np.random.seed(SEED)
sc.settings.verbosity = 1

## Step 1 — load the TCR-complete object, find the dominant CD4 clone, gate the cohort · HEAVY

Loads `skin_T_annotated.h5ad` and folds in the Li2024 V(D)J parquet (~12 GB cache, built once).
Dominance is judged **on CD4 only** (`dom_mask`) so a co-expanded reactive CD8 clone cannot win the
"top clone" contest in a donor whose infiltrate is larger than its tumour.

Two gates, because the two halves of this notebook need different things from a donor:

| gate | who it selects | rule |
|---|---|---|
| `keep_donors` | everything downstream (incl. inferCNV, Steps 9–15) | ≥ `CNV_MIN_T` T cells |
| `tcr_donors`  | the ALICE clonality call (Steps 2–4) | ≥ `TCR_MIN_CELLS` TCR+ CD4 cells |

inferCNV needs no repertoire, so seven v2 cohorts with no V(D)J at all (brentuximab26, lyp26,
rindler21mc, alkon24, gaydosik19, gaydosik23, jonak21) stay in the object and get a CNV score;
`obs["tcr_cohort"]` marks them so nothing downstream reads "no TCR data" as "TCR-negative".

In [ ]:
adata = C.build_or_load_tcr_object(OBJ, TCR_OBJ, LI_TCR, H)   # ~12 GB cache built on first run
adata.obs["cell_type_T2"] = adata.obs["cell_type_T"].astype(str)   # shared-helper alias
is_cd4 = adata.obs["cell_type_T2"].isin(CD4_T_TYPES).to_numpy()
is_li  = adata.obs["study"].astype(str).eq("li2024").to_numpy()
print("cell_type_T:\n", adata.obs["cell_type_T"].value_counts())

# Unified TRB clone key + per-donor dominant clone, decided within CD4 only. Sets obs:
# tcr_clone_id / tcr_clone_size / tcr_is_expanded / tcr_is_dominant_clone / tcr_is_malignant.
dom_tbl = C.recompute_dominant_clone(adata, H, is_li, FRAC_THRESH, RATIO_THRESH, EXPANDED_MIN,
                                     dom_mask=is_cd4)
# One row per (donor, TRB CDR3) over CD4 cells — the repertoire ALICE runs on. Built before the
# gates so the TCR gate can read its cell counts straight off it.
clono_cd4 = A.clonotype_table(adata.obs[is_cd4], group="donor")

META_COLS = ["study", "dataset", "disease", "disease_stage", "stage_class", "entity", "tissue",
             "tissue_detail", "lesion_type", "cohort_country", "cohort_region",
             "treatment_context", "blood_involvement", "sex", "tech"]
meta_cols = [c for c in META_COLS if c in adata.obs.columns]
donor_meta = (adata.obs[["donor", *meta_cols]].astype(str)
              .groupby("donor", observed=True)
              .agg(lambda s: ", ".join(sorted(s[s != "nan"].unique()))))

# ---- gate 1: keep_donors (everything, incl. CNV) ----
n_T = adata.obs["donor"].astype(str).value_counts()
drop = {}
if {"D5__MFIVB", "D1__P303"} <= set(donor_meta.index):
    drop["D1__P303"] = "duplicate of D5__MFIVB"
for d in donor_meta.index[donor_meta["entity"].isin(DROP_ENTITIES)]:
    drop.setdefault(d, "non-ab-CD4 entity")
for d in n_T.index[n_T < CNV_MIN_T]:
    drop.setdefault(str(d), f"n_T < {CNV_MIN_T}")
keep_donors = [d for d in donor_meta.index if d not in drop]

# ---- gate 2: tcr_donors (ALICE only). herrera2021 is exempt (small but deeply sequenced); HC
# donors are exempt because they are never query patients — they are the diploid CNV reference plus
# the held-out negative control (Step 9), so repertoire depth is irrelevant to their role. ----
_meta = donor_meta.loc[keep_donors]
n_tcr_cd4 = (clono_cd4.groupby("donor", observed=True)["n_cells"].sum()
             .reindex(keep_donors).fillna(0).astype(int))
_exempt = _meta["study"].eq("herrera2021") | _meta["disease"].eq("HC")
tcr_donors = sorted(n_tcr_cd4.index[(n_tcr_cd4 >= TCR_MIN_CELLS) | _exempt])

adata = adata[adata.obs["donor"].isin(keep_donors)].copy()
adata.obs["tcr_cohort"] = adata.obs["donor"].astype(str).isin(tcr_donors)
clono_cd4 = clono_cd4[clono_cd4["donor"].isin(tcr_donors)].reset_index(drop=True)

print("dropped donors:", drop)
print(f"kept: {len(keep_donors)} donors / {adata.n_obs:,} T cells / "
      f"{adata.obs['sample_id'].nunique()} samples | HC donors: {int(_meta['disease'].eq('HC').sum())}")
print(f"TCR (ALICE) cohort: {len(tcr_donors)} donors / {int(adata.obs['tcr_cohort'].sum()):,} cells"
      f" | CNV-only: {len(keep_donors) - len(tcr_donors)} donors /"
      f" {int((~adata.obs['tcr_cohort']).sum()):,} cells")
print("CD4 clonotypes:", len(clono_cd4), "| donors:", clono_cd4["donor"].nunique(),
      "| donors with a dominant clone:", int(clono_cd4.groupby("donor")["is_founder"].any().sum()))

## Step 2 — OLGA generative null

`A.load_olga_trb()` loads the default human-TRB IGoR model. Expected ≤1-aa neighbors of a clonotype =
`N_unique × Q × Σ Pgen(single-mismatch variants)`; observed vs expected is a Poisson survival test,
BH-corrected per repertoire. **Requires OLGA** — run `%pip install olga` once in this kernel.

In [ ]:
# %pip install olga    # uncomment + run once if olga is not importable
pgen_model = A.load_olga_trb()
pgen = A.make_pgen(pgen_model)

# sanity: the Rindler IVB single-aa-variant pair (a textbook ALICE edge) must form a graph edge
_a, _b = "CASSQDRALENTIYF", "CASSQDRTLENTIYF"
assert _b in set(A.one_mismatch_variants(_a)) and A.neighbor_graph([_a, _b]).has_edge(_a, _b)
print(f"OLGA ready | pgen({_a}) = {pgen(_a):.2e}")

## Step 3 — ALICE on the CD4 repertoire → the per-cell clonality call · HEAVY (OLGA sweep)

One ALICE run, per donor, on the pooled CD4 repertoire. For each donor:

1. **seed** = the dominant CD4 clone(s); if the donor has none, its largest clonotype
   (`founder_source` records which — the second case is what lets ALICE *recover* a tumour whose
   cells are split across β-variants and so never looked dominant as an exact clone).
2. **family** = the ≤1-aa connected component of the seed, kept to clonotypes ALICE called
   significant against the OLGA null, plus the seed itself.
3. **donor call** = family ≥ `FRAC_THRESH` of the donor's TCR+ CD4 **and** ≥ `RATIO_THRESH` × its
   largest non-family clone.

`obs["tcr_clonal"]` is then the headline per-cell column: this CD4 cell carries the founder family
of a donor that passed. `tcr_is_malignant` / `tcr_is_dominant_clone` / `tcr_malignant_alice` are set
from it, which is the contract Steps 9–15 read.

*(Removed in this rewrite: the cross-patient convergence test — 0 significant hits out of 30 pooled
founders, an explicit negative control — and the reactive-CD8 ALICE pass, which answers a different
question about the TIL infiltrate. Both are in git history before this commit if they are needed.)*

In [ ]:
alice_cd4 = pd.read_parquet(ALICE_CD4_PARQUET) if ALICE_CD4_PARQUET.exists() else None
if alice_cd4 is None or set(alice_cd4["donor"]) != set(clono_cd4["donor"]):
    if alice_cd4 is not None:
        print("ALICE cache stale (donor set changed) -> recompute")
    alice_cd4 = A.run_alice_by_group(clono_cd4, pgen, group="donor", Q=Q, alpha=ALPHA)
    alice_cd4.to_parquet(ALICE_CD4_PARQUET, index=False)
    print("wrote", ALICE_CD4_PARQUET)
else:
    print("loaded cached", ALICE_CD4_PARQUET)
print("ALICE donors:", alice_cd4["donor"].nunique(),
      "| significant clonotypes:", int(alice_cd4["significant"].sum()))

# PT35's tumour is split across two co-dominant clones, neither of which wins the dominance test on
# its own; seeding from both is a documented, donor-specific exception carried from v1.
PT35_EXCEPTION = "Li2024_atlas__PT35"

# Herrera's SS donors are annotated MF upstream; relabel before anything strata by disease, and
# before the donor call below reads `disease` to force HC benign.
donor_of = adata.obs["donor"].astype(str)
_ss = donor_of.isin([f"H__SS{i}" for i in range(1, 7)]).to_numpy()
if _ss.any():
    adata.obs["disease"] = adata.obs["disease"].astype(str)
    adata.obs.loc[_ss, "disease"] = "SS"
    adata.obs["disease"] = adata.obs["disease"].astype("category")
    print("relabeled Herrera SS donors -> disease='SS':", int(_ss.sum()), "cells")
HC_DONORS = set(donor_of[adata.obs["disease"].astype(str).eq("HC").to_numpy()])

fam_by_donor, rows = {}, []
for donor, sub in clono_cd4.groupby("donor", observed=True):
    sig = set(alice_cd4.loc[(alice_cd4["donor"] == donor) & alice_cd4["significant"], "cdr3"])
    if donor == PT35_EXCEPTION:
        seeds, src = set(sub.nlargest(2, "n_cells")["cdr3"]), "pt35_top2"
    elif sub["is_founder"].any():
        seeds, src = set(sub.loc[sub["is_founder"], "cdr3"]), "dominant"
    else:
        seeds, src = {sub.nlargest(1, "n_cells")["cdr3"].iloc[0]}, "largest_clonotype"
    family = seeds | (A.founder_family(sub, seeds=seeds) & sig)
    fam_by_donor[donor] = family

    in_fam = sub["cdr3"].isin(family)
    n_fam, n_tot = int(sub.loc[in_fam, "n_cells"].sum()), int(sub["n_cells"].sum())
    rest = sub.loc[~in_fam, "n_cells"]
    n_next = int(rest.max()) if len(rest) else 0
    frac = n_fam / n_tot if n_tot else 0.0
    fold = (n_fam / n_next) if n_next else np.inf
    rows.append({"donor": donor, "founder_cdr3": "; ".join(sorted(seeds)), "founder_source": src,
                 "n_family_clones": len(family), "n_family_significant": len(family & sig),
                 "donor_family_frac": frac, "donor_family_fold": fold,
                 # HC is benign by definition — a healthy donor's largest clone is not a tumour
                 # clone however dominant it looks, and this is the one place to say so, so that
                 # obs["tcr_clonal"] and the per-sample table cannot disagree about it.
                 "donor_is_clonal": (donor not in HC_DONORS)
                                    and (frac >= FRAC_THRESH) and (fold >= RATIO_THRESH)})
DONOR_FAM = pd.DataFrame(rows).set_index("donor")

# ---- map the family back to cells ----
# TRB comes from the unified clone key (Li2024 cells carry theirs only there), as in clonotype_table.
is_cd4 = adata.obs["cell_type_T2"].isin(CD4_T_TYPES).to_numpy()
trb = (adata.obs["tcr_clone_id"].astype(str)
       .str.extract(r"^TRB:([A-Z]+)$", expand=False).fillna(""))
trb = trb.mask(trb == "", adata.obs["trb_cdr3"].astype(str))

clonal = np.zeros(adata.n_obs, dtype=bool)
for dn, family in fam_by_donor.items():
    if not DONOR_FAM.loc[dn, "donor_is_clonal"]:
        continue                      # a polyclonal donor's largest clone is not a tumour clone
    clonal |= is_cd4 & (donor_of == dn).to_numpy() & trb.isin(family).to_numpy()

# the headline column, plus the three aliases Steps 9-15 read
adata.obs["tcr_clonal"] = clonal
adata.obs["tcr_malignant_alice"]   = clonal
adata.obs["tcr_is_malignant"]      = clonal
adata.obs["tcr_is_dominant_clone"] = clonal
pd.DataFrame({"cell_id": adata.obs_names, "tcr_clonal": clonal}).to_parquet(ALICE_MAL_PARQUET,
                                                                            index=False)
print(f"\nclonal donors: {int(DONOR_FAM['donor_is_clonal'].sum())}/{len(DONOR_FAM)}"
      f" | founder source: {DONOR_FAM['founder_source'].value_counts().to_dict()}")
print(f"clonal CD4 cells: {int(clonal.sum()):,} / {int(is_cd4.sum()):,}")
print("wrote", ALICE_MAL_PARQUET)

## Step 4 — the conclusive per-sample clonality table

One row per sample: **is it clonal, how many CD4 cells, what fraction of them are clonal**, plus the
descriptive fields needed to stratify (study, disease, stage, site, lesion type, origin, treatment).

`family_frac` and `family_fold` are recomputed **within each sample** — against that sample's own
TCR+ CD4 cells and its own largest non-family clone — so `is_clonal` is a statement about the
sample, not the donor's call copied down every row. A sample is clonal when its donor is clonal
**and** the family clears both thresholds locally; a biopsy where the tumour clone is absent
therefore reads False even in a clonal patient.

Samples whose donor has no repertoire at all (`tcr_cohort=False`) get `is_clonal = <NA>`, never
False — no TCR data is absence of evidence, not evidence of absence.

In [ ]:
cd4 = pd.DataFrame({
    "sample_id": adata.obs["sample_id"].astype(str).to_numpy(),
    "donor": donor_of.to_numpy(),
    "cdr3": trb.to_numpy(),
    "has_tcr": adata.obs["has_tcr"].astype(bool).to_numpy(),
    "clonal": clonal,
})[is_cd4]
cd4["tcr_ok"] = cd4["has_tcr"] & cd4["cdr3"].map(A._valid)     # same validity rule as clonotype_table

rows = []
for sid, sub in cd4.groupby("sample_id", observed=True):
    dn = sub["donor"].iat[0]
    fam = fam_by_donor.get(dn, set())
    vc = sub[sub["tcr_ok"]].groupby("cdr3", observed=True).size()
    in_fam = vc.index.isin(fam)
    n_fam, n_rep, n_clonal = int(vc[in_fam].sum()), int(vc.sum()), int(sub["clonal"].sum())
    n_next = int(vc[~in_fam].max()) if (~in_fam).any() else 0
    rows.append({
        "sample_id": sid, "donor": dn,
        "n_cd4": len(sub), "n_cd4_tcr": n_rep, "n_clonal": n_clonal,
        "frac_clonal": (n_clonal / n_rep) if n_rep else np.nan,
        "frac_clonal_of_cd4": n_clonal / len(sub),
        "family_frac": (n_fam / n_rep) if n_rep else np.nan,
        "family_fold": (n_fam / n_next) if n_next else (np.inf if n_fam else np.nan)})
smp = pd.DataFrame(rows)

meta_by_sample = (adata.obs[["sample_id", *meta_cols]].astype(str)
                  .groupby("sample_id", observed=True)
                  .agg(lambda s: ", ".join(sorted(s[s != "nan"].unique())))
                  .reset_index())
n_smp_donor = adata.obs.groupby("donor", observed=True)["sample_id"].nunique()

clon = (smp.merge(DONOR_FAM.reset_index(), on="donor", how="left")
        .merge(meta_by_sample, on="sample_id", how="left")
        .assign(n_samples_in_donor=lambda d: d["donor"].map(n_smp_donor).astype(int),
                tcr_cohort=lambda d: d["donor"].isin(tcr_donors)))

# clonal = the donor is clonal AND the family clears both thresholds in THIS sample. A sample with
# no repertoire of its own is <NA>, never False.
clon["is_clonal"] = ((clon["donor_is_clonal"].eq(True)
                      & clon["family_frac"].ge(FRAC_THRESH)
                      & clon["family_fold"].ge(RATIO_THRESH))
                     .astype("boolean")
                     .mask(~clon["tcr_cohort"] | clon["n_cd4_tcr"].eq(0)))

COLS = (["sample_id", "donor", "n_samples_in_donor", "tcr_cohort",
         "is_clonal", "n_cd4", "n_cd4_tcr", "n_clonal", "frac_clonal", "frac_clonal_of_cd4",
         "family_frac", "family_fold", "founder_cdr3", "founder_source",
         "n_family_clones", "n_family_significant", "donor_is_clonal", "donor_family_frac"]
        + meta_cols)
clon = clon[[c for c in COLS if c in clon.columns]].sort_values(
    ["is_clonal", "frac_clonal"], ascending=False, na_position="last").reset_index(drop=True)
clon.to_csv(CLONALITY_CSV, index=False)

_cl = clon["is_clonal"]
print("wrote", CLONALITY_CSV, clon.shape)
print(f"samples: {len(clon)} | clonal {int(_cl.eq(True).fillna(False).sum())}"
      f" | not clonal {int(_cl.eq(False).fillna(False).sum())}"
      f" | no TCR data {int(_cl.isna().sum())}")
print(f"CD4 cells: {int(clon['n_cd4'].sum()):,} | TCR+ {int(clon['n_cd4_tcr'].sum()):,}"
      f" | clonal {int(clon['n_clonal'].sum()):,}")
print("\nclonal rate by disease x stage:")
print(clon.assign(_c=_cl.astype("float"))
      .pivot_table(index="disease", columns="stage_class", values="_c", aggfunc="mean")
      .round(2).to_string())
clon

## Step 4b — what "clonal" looks like: the TRB distribution of one sample

Illustration, not an input to any call: one clonal sample's CD4 repertoire, top clonotypes vs the rest.

In [ ]:
# Top TRB clonotypes of one clonal sample (illustration only — reads in-memory objects).
SHOWCASE_SAMPLE = None     # set a sample_id to override the automatic pick
TOP_N = 10

_cl = clon[clon["is_clonal"].eq(True) & clon["n_cd4_tcr"].ge(200)]
SID = SHOWCASE_SAMPLE or _cl.sort_values("family_frac", ascending=False)["sample_id"].iat[0]
row = clon[clon["sample_id"].eq(SID)].iloc[0]
seeds = {s for s in str(DONOR_FAM.loc[row["donor"], "founder_cdr3"]).split("; ") if s}

vc = (cd4[cd4["sample_id"].eq(SID) & cd4["tcr_ok"]]
      .groupby("cdr3", observed=True).size().sort_values(ascending=False))
pct = vc / vc.sum() * 100
top = pct.head(TOP_N)
bars = pd.concat([top, pd.Series({f"all other clonotypes (n={len(pct) - TOP_N:,})":
                                  pct.iloc[TOP_N:].sum()})])[::-1]
colors = ["#C1443C" if c in seeds else "#B9BDC2" if c.startswith("all other") else "#3B6FB6"
          for c in bars.index]

fig, ax = plt.subplots(figsize=(6, 3.6), dpi=130)
ax.barh(np.arange(len(bars)), bars.to_numpy(), height=0.7, color=colors)
for y, v in enumerate(bars.to_numpy()):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=8, color="#33373B")
ax.set(yticks=np.arange(len(bars)), xlabel="% of sample CD4 TRB repertoire",
       xlim=(0, bars.max() * 1.18))
ax.set_yticklabels(bars.index, fontsize=7.5, family="monospace")
ax.set_title(f"{SID}: one CD4 clone is {top.iloc[0]:.0f}% of the repertoire\n"
             f"({int(vc.sum()):,} TCR+ cells, {len(pct):,} clonotypes)", fontsize=10, loc="left")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / f"skin_tcr_clone_distribution_{SID}.png", dpi=200, bbox_inches="tight",
            facecolor="white")
plt.show()

## Step 5 — inferCNV inputs: query + pooled diploid reference · HEAVY (GPU kernel)

The CNV arm is the **null-anchored, de novo signed arm-consensus caller** — developed in `32_malignancy_tcr_cnv` on
blood (v3.1) and validated on skin as v4 (held-out healthy control 3.1 % false positive, median
per-donor AUROC 0.807 vs the TCR call). It is the method, not a choice among methods, so nothing
below is tunable; the earlier knob panel, the focal-cluster / union / legacy-GMM channels and the
method-selection QC tables are gone (recoverable from git history).

`C.prepare_skin_cnv_inputs` builds the query (annotated T of disease donors) and a **pooled**
diploid reference of three categories whose biases partly cancel:

| `cnv_ref` | cells | bias | role |
|---|---|---|---|
| `nonclonal` | within-donor non-dominant T | right batch + lineage, **tumour-contaminated** | scored like any query cell, **not** a baseline |
| `cd8_ref` | same-donor non-dominant CD8 (≤2000/donor) | right batch, wrong lineage | baseline; **half held out as the batch-matched null** |
| `hc_atlas` | atlas HC T | right tissue + lineage, few donors | baseline; ⅔ used, ⅓ held out |
| `healthy` | external healthy/AD/pso benign-T | right lineage, many donors, wrong batch | baseline |

`cd8_ref` lives inside `acnv` per donor; `hc_atlas` / `healthy` come back in `SHARED_REF`. The
held-out HC control (`HC_CONTROL`, ⅓ of the atlas HC T) stays in the query and is the run's
acceptance test — a working caller must leave it near zero. Self-normalises from `raw_counts`,
adds genomic positions from the GTF, restricts to chr1-22/X/Y.

In [ ]:
import os

# ---- the estimator, fixed. Splatted into BOTH inferCNV passes below: they must hold out exactly
# the same null cells, or the arm null describes different diploid cells than the score null.
# The helper *defaults* are the superseded v2/v3 estimator (ref_mode="bounded",
# dynamic_threshold=1.5, shuffle=False, null_frac=0, window=250), so these overrides have to stay
# explicit — dropping one silently reproduces the failing estimator.
#   window=100          skin's ~15.4k positioned genes make v3's 250 too coarse (~1.6% of genome)
#   ref_mode="mean"     one pooled reference vector; infercnvpy's >=2-category path silently
#                       switches to a bounded dead-band that suppresses real signal
#   dynamic_threshold   None: the 1.5*std(chunk) zeroing depended on chunk composition, which made
#                       scores incomparable across donors
#   shuffle=True        permute before chunking so every chunk has the same query/ref mix
#   null_frac=0.5       half the baseline is held out and scored as query -> the empirical null
#   null_cats           the in-object half of the null is batch-matched same-donor CD8
CNV_EST  = dict(window=100, ref_mode="mean", dynamic_threshold=None, shuffle=True,
                null_frac=0.5, null_cats=("cd8_ref",), seed=SEED)
REF_CATS = ("hc_atlas", "cd8_ref", "healthy")   # nonclonal deliberately absent (tumour-contaminated)

CNV_CACHE  = OUT_DIR / "skin_T_cnv_per_cell_v5_res0.5.parquet"
ARM_CACHE  = OUT_DIR / "skin_T_arm_cnv_v5.parquet"
CNV_N_JOBS = 8                              # score pass: 15.4k genes densify, keep memory bounded
ARM_JOBS   = len(os.sched_getaffinity(0))   # arm pass is chunked at 100; respects the LSF cpuset

acnv, SHARED_REF, CNV_DONORS, HC_CONTROL = C.prepare_skin_cnv_inputs(
    adata, GTF, INTEGRATED_H5, SEED,
    n_healthy_ref=10000, hc_ref_frac=2 / 3, cd8_ref_cap=2000)
print("query donors:", len(CNV_DONORS), "| acnv:", acnv.shape,
      "| query:", int((acnv.obs["cnv_ref"] == "query").sum()),
      "| held-out HC control cells:", len(HC_CONTROL))

In [ ]:
# ---- 5a. per-donor inferCNV vs the pooled reference -> per-cell scores + CNV Leiden clusters.
# Returns the full per-cell frame INCLUDING the held-out `ref_null` rows; Step 6 needs them.
PER_CELL = C.run_per_donor_infercnv(
    acnv, SHARED_REF, CNV_DONORS, CNV_CACHE, **CNV_EST, ref_cats=REF_CATS,
    topk_frac=0.10, leiden_res=0.5, depth_col="n_genes_cnv",
    n_jobs=CNV_N_JOBS, chunk=1000, force=False)

print("\nper-cell rows:", PER_CELL.shape)
print(PER_CELL["cnv_ref"].value_counts().to_string())
print("\nnull cells per donor (need >= 50 for a usable threshold):")
print(PER_CELL[PER_CELL.cnv_ref.eq("ref_null")].groupby("donor", observed=True)
      .size().describe()[["min", "25%", "50%", "max"]].round(0).to_string())

In [ ]:
# ---- 5b. second pass with `calculate_gene_values=True`, averaged per chromosome arm -> the
# cells x ~41 arm matrix the caller is built on. Same `**CNV_EST`, so `_split_null` holds out
# exactly the same cells as the score pass. Checkpoints per donor in `<cache>_parts/`.
arm = C.compute_arm_cnv_per_cell(
    acnv, CNV_DONORS, ARM_CACHE, shared_ref=SHARED_REF, **CNV_EST, reference_cat=REF_CATS,
    keep_cats=("query", "nonclonal", "ref_null"), n_jobs=ARM_JOBS, chunk=100, force=False)
ARM_COLS = [c for c in arm.columns if c != "donor"]
print("wrote", ARM_CACHE, "| arm matrix:", arm[ARM_COLS].shape, "| arms:", ARM_COLS)

_null_arm = set(arm.index[arm.index.astype(str).str.contains(r"\|NULL\|")])
_null_pc = set(PER_CELL.index[PER_CELL.cnv_ref.eq("ref_null")])
print(f"shared-reference null cells — arm pass {len(_null_arm)}, score pass "
      f"{len(_null_pc & set(arm.index))}: {_null_arm <= _null_pc}")
assert _null_arm <= _null_pc, "the two inferCNV passes held out different reference cells"

## Step 6 — the malignancy call

`cnv_focal_score` is `mean|X_cnv|` over the top windows, so scattered dropout noise scores like a
clean chr7q gain. A real event is **signed**, contiguous and **shared across the clone**, so
`C.arm_consensus_score` tests each donor's own CNV clusters arm-by-arm against that donor's
held-out diploid null:

1. per arm, the null's mean and sd over the donor's `ref_null` cells;
2. a cluster carries an event on an arm when it is ≥5 SE from the null mean **and** shifted by
   ≥0.01 in absolute terms;
3. a candidate cluster must hold ≥ `max(30, 5 % of the donor's T)` cells — a signature fitted to
   ~40 cells is fitted to their noise;
4. a cluster shifted on >70 % of arms is a global offset, not a karyotype, and is rejected; one
   shifted on >15 arms keeps its 15 strongest (Sézary karyotypes really do carry 13–15);
5. the largest surviving cluster defines the donor's signed signature;
6. every cell scores as its projection onto that unit signature, cut at the depth-conditioned
   98th percentile of the same projection over the null cells.

**No TCR input anywhere**, so the CNV call stays independent evidence. A donor with no cluster
clearing the null gets no signature and no calls — `indeterminate`, not silently negative.

In [ ]:
CD4 = acnv.obs["cnv_ref"].isin(["query", "nonclonal"]).to_numpy()   # every scored T cell
PT_DONORS = sorted(acnv.obs.loc[CD4, "donor"].astype(str).unique())

arm_proj, arm_call, arm_state, SIG = C.arm_consensus_score(
    arm, PER_CELL, PT_DONORS, arm_cols=ARM_COLS, cluster_col="cnv_leiden",
    depth_col="n_genes_cnv", q=0.98, min_cluster_frac=0.05,
    max_event_arms=15, global_arm_frac=0.7)

acnv.obs["cnv_arm_proj"] = (pd.Series(arm_proj, index=arm.index)
                            .reindex(acnv.obs_names).to_numpy(dtype=float))
acnv.obs["cnv_arm_malignant"] = (pd.Series(arm_call, index=arm.index)
                                 .reindex(acnv.obs_names).eq(True).to_numpy() & CD4)
acnv.obs["cnv_state"] = (pd.Series(arm_state, index=arm.index).reindex(acnv.obs_names)
                         .fillna("indeterminate").astype(str))

# ---- propagate to `adata`. A donor is callable iff the arm pass found a signature for it. ----
CALLABLE = set(SIG.loc[SIG["callable"], "donor"].astype(str))
for col in ["cnv_score", "cnv_cell_score", "cnv_focal_score", "cnv_arm_proj"]:
    adata.obs[col] = acnv.obs[col].where(CD4).reindex(adata.obs_names)
adata.obs["cnv_leiden"] = acnv.obs["cnv_leiden"].reindex(adata.obs_names).fillna("")
adata.obs["cnv_state"] = (acnv.obs["cnv_state"].reindex(adata.obs_names)
                          .fillna("indeterminate").astype(str))
adata.obs["cnv_arm_malignant"] = (acnv.obs["cnv_arm_malignant"]
                                  .reindex(adata.obs_names).eq(True).to_numpy())
adata.obs["cnv_callable"] = adata.obs["donor"].astype(str).isin(CALLABLE).to_numpy()

_hc = adata.obs_names.isin(HC_CONTROL)
print(f"\ncallable donors: {len(CALLABLE)}/{len(PT_DONORS)}  |  "
      f"malignant {int(adata.obs['cnv_arm_malignant'].sum())}/{adata.n_obs} cells")
print(f"held-out HC control: {int(adata.obs.loc[_hc, 'cnv_arm_malignant'].sum())}/{int(_hc.sum())}"
      f" called malignant ({adata.obs.loc[_hc, 'cnv_arm_malignant'].mean():.1%})"
      f"   [gate <= 5%; v4/v1-cohort was 3.1%]")
SIG

## Step 7 — inferCNV heatmaps for a few donors · HEAVY (GPU kernel)

A visual check on the call: three high-burden callable donors, one `indeterminate` donor (should
read flat, or flat-plus-noise), and one held-out HC-control donor (must read flat). Picked from
the result rather than hardcoded, since the v2 re-annotation changed the donor set.

Drawn with the **same estimator** that produced the call (`window=100`, pooled-mean reference),
so the picture is the matrix the caller actually saw. Payloads cached under
`cnv_heatmap_cache_v5`, so re-plots are free.

In [ ]:
_d = adata.obs[adata.obs["cnv_focal_score"].notna()].copy()
_d["pt"] = _d["donor"].astype(str)
_g = _d.groupby("pt", observed=True).agg(frac=("cnv_arm_malignant", "mean"),
                                         n=("cnv_arm_malignant", "size"),
                                         ok=("cnv_callable", "first"))
_hc_d = sorted(_d.loc[_d.index.isin(HC_CONTROL), "pt"].unique())
_pool = _g[(_g["n"] >= 500) & ~_g.index.isin(_hc_d)]
HEATMAP_DONORS = (_pool[_pool["ok"]].sort_values("frac", ascending=False).head(3).index.tolist()
                  + _pool[~_pool["ok"]].sort_values("n", ascending=False).head(1).index.tolist()
                  + _hc_d[:1])
print("heatmap donors:", HEATMAP_DONORS)

CNV_HM_CACHE = OUT_DIR / "cnv_heatmap_cache_v5"
sub = acnv[acnv.obs["donor"].astype(str).isin(HEATMAP_DONORS).to_numpy()].copy()
if sub.n_obs:
    C.run_cnv_heatmaps(
        sub, SHARED_REF, call_col="cnv_arm_malignant", reference_cat=REF_CATS,
        ref_mode=CNV_EST["ref_mode"], dynamic_threshold=CNV_EST["dynamic_threshold"],
        window=CNV_EST["window"], fig_prefix="skin_T_v5_cnv_heatmap", fig_dir=FIG_DIR,
        n_per_study=len(HEATMAP_DONORS), vlim_scale=3.0, cmap="RdBu_r",
        order_by="cnv_leiden", n_jobs=CNV_N_JOBS, chunk=1000,
        cnv_cache_dir=CNV_HM_CACHE, force_cnv=False)
del sub
gc.collect()

## Step 8 — combine TCR + CNV & persist

`combined_malignant = TCR ∨ (CNV ∧ donor is callable)`. In an `indeterminate` donor the CNV caller
found no clone-level event clearing the diploid null, so a negative there is absence of evidence,
not evidence of absence — those cells fall back to the TCR label and are tagged
`*_cnv_indeterminate`. Symmetrically, in the seven v2 cohorts with no V(D)J a TCR negative means
*no repertoire*, so those cells get their own `*_no_tcr_data` levels and no downstream reader can
mistake missing TCR for a negative TCR.

Persisted per-cell to `skin_T_malignancy_v5.parquet`.

In [ ]:
tcr_m  = adata.obs["tcr_is_malignant"].to_numpy()
cnv_m  = adata.obs["cnv_arm_malignant"].to_numpy().astype(bool)
cnv_ok = adata.obs["cnv_callable"].to_numpy()
has_tcr_arm = adata.obs["tcr_cohort"].to_numpy()

adata.obs["combined_malignant"] = tcr_m | (cnv_m & cnv_ok)
adata.obs["malignant_evidence"] = np.select(
    [tcr_m & cnv_m & cnv_ok, tcr_m & ~cnv_ok, tcr_m,
     ~tcr_m & cnv_m & cnv_ok & has_tcr_arm, ~tcr_m & cnv_m & cnv_ok & ~has_tcr_arm,
     ~cnv_ok & ~has_tcr_arm, ~cnv_ok, ~has_tcr_arm],
    ["both", "tcr_only_cnv_indeterminate", "tcr_only",
     "cnv_only", "cnv_only_no_tcr_data",
     "none_no_tcr_data_cnv_indeterminate", "none_cnv_indeterminate", "cnv_negative_no_tcr_data"],
    default="none")
print("combined malignant:", int(adata.obs["combined_malignant"].sum()), "/", adata.n_obs)
print("\nevidence:\n", adata.obs["malignant_evidence"].value_counts())

# TPR/FPR are only meaningful where both channels exist: TCR-cohort donors with a callable CNV run.
both_avail = (adata.obs["has_tcr"].to_numpy() & has_tcr_arm & cnv_ok
              & adata.obs["cnv_focal_score"].notna().to_numpy())
ct = pd.crosstab(adata.obs.loc[both_avail, "tcr_is_malignant"],
                 adata.obs.loc[both_avail, "cnv_arm_malignant"], rownames=["tcr"], colnames=["cnv"])
if ct.shape == (2, 2):
    print(f"\nvs the TCR call on {int(both_avail.sum())} cells with both signals:  "
          f"TPR={ct.loc[True, True] / ct.loc[True].sum():.2f}  "
          f"FPR={ct.loc[False, True] / ct.loc[False].sum():.2f}   [v4/v1-cohort was 0.64 / 0.07]")

keep = ["donor", "study", "disease", "cell_type", "cell_type_T2", "cached_malignant",
        "tcr_cohort", "has_tcr", "tcr_clone_id", "tcr_clone_size", "tcr_is_expanded",
        "tcr_is_dominant_clone", "tcr_is_malignant", "tcr_clonal", "tcr_malignant_alice",
        "cnv_score", "cnv_cell_score", "cnv_focal_score", "cnv_arm_proj", "cnv_leiden",
        "cnv_arm_malignant", "cnv_state", "cnv_callable",
        "combined_malignant", "malignant_evidence"]
out = adata.obs[[c for c in keep if c in adata.obs.columns]].copy()
out.index.name = "obs_name"
out.to_parquet(OUT_PARQUET)
print("\nwrote", OUT_PARQUET, out.shape)

## Step 9 — conclusive per-donor CNV table

One row per donor with a CNV run: cohort size, the TCR-clonal fraction where a repertoire exists,
the donor's called state and signed arm signature, and the fraction of its T cells called
malignant. `auroc_vs_tcr` is the projection's discrimination against the TCR call, reported only
for donors with ≥50 TCR-positive **and** ≥50 TCR-negative cells (AUROC at ~100 % prevalence is
meaningless); CNV-only donors have no TCR truth and are left `NaN`.

The three lines under the table are the run's acceptance check.

In [ ]:
from sklearn.metrics import roc_auc_score

qdf = adata.obs[adata.obs["cnv_focal_score"].notna()].copy()
qdf["pt"] = qdf["donor"].astype(str)
rows = []
for pt, x in qdf.groupby("pt", observed=True):
    t = x[x["has_tcr"].to_numpy()]
    yt = t["tcr_is_malignant"].to_numpy().astype(bool)
    balanced = yt.sum() >= 50 and (~yt).sum() >= 50
    s = t["cnv_arm_proj"].to_numpy(dtype=float)
    ok = np.isfinite(s)
    rows.append({
        "donor": pt, "study": str(x["study"].iat[0]), "n_T": len(x), "n_tcr": len(t),
        "tcr_malig_frac": round(float(yt.mean()), 3) if len(t) else np.nan,
        "cnv_state": "callable" if bool(x["cnv_callable"].iat[0]) else "indeterminate",
        "frac_cnv_malignant": round(float(x["cnv_arm_malignant"].mean()), 3),
        "auroc_vs_tcr": (round(float(roc_auc_score(yt[ok], s[ok])), 3)
                         if balanced and ok.sum() > 100 and yt[ok].any() and (~yt[ok]).any()
                         else np.nan),
        "is_hc_control": bool(x.index.isin(HC_CONTROL).any())})

# `anchor_frac` is only written for callable donors, so it is absent from SIG entirely
# if nothing was callable — take whatever columns are there.
_sig_cols = [c for c in ["donor", "n_event_arms", "anchor_frac", "signature", "reason"]
             if c in SIG.columns]
summary = (pd.DataFrame(rows).merge(SIG[_sig_cols], on="donor", how="left")
           .sort_values("frac_cnv_malignant", ascending=False))
summary.to_csv(SUMMARY_CSV, index=False)
print("wrote", SUMMARY_CSV, summary.shape)

_hc = adata.obs_names.isin(HC_CONTROL)
_bal = summary["auroc_vs_tcr"].notna()
print(f"\n1. held-out HC control: {int(adata.obs.loc[_hc, 'cnv_arm_malignant'].sum())}"
      f"/{int(_hc.sum())} called malignant "
      f"({adata.obs.loc[_hc, 'cnv_arm_malignant'].mean():.1%})   [gate <= 5%; v4 was 3.1%]")
print(f"2. median per-donor AUROC vs TCR {summary.loc[_bal, 'auroc_vs_tcr'].median():.3f} "
      f"(n={int(_bal.sum())} balanced)  [v4 was 0.807]  |  callable "
      f"{int(summary.cnv_state.eq('callable').sum())}/{len(summary)}  |  donors called 0%/100%: "
      f"{int(summary.frac_cnv_malignant.eq(0).sum())}/{int(summary.frac_cnv_malignant.eq(1).sum())}")
print("3. recurrent signed arm events (expect chr17p-, chr10q-, chr8q+, chr7p+; a chr19q- at the "
      "top is a suspected gene-density artifact):")
print(SIG.loc[SIG["callable"], "signature"].str.split(", ").explode().value_counts()
      .head(12).to_string())
summary